# The Du et al. (2025) refinery planning benchmark

W. Du, C. Wang, C. Fan et al., "A production planning benchmark for real-world refinery-petrochemical complexes", arXiv:2503.22057 (2025), publishes a refinery topology with crude data, prices, product specifications and a delta-base planning model. `difflow_refinery.benchmarks.du2025` implements its Case 1 as published (#530): two swing-cut crude units, 20 fixed-yield units, 5 delta-base units, 32 mixers, 61 splitters and 12 blenders. They are assembled into one differentiable JAX model.

This notebook:

1. loads the data and builds the model;
2. checks the published optimum against it: every equation, every limit, and the profit;
3. re-solves the model locally with pounce;
4. reads the binding capacities off the multipliers;
5. differentiates a delta-base unit, which is the Jacobian that later work compares against nonlinear units.

The data are **not** shipped with difflow, because the paper is CC BY-NC-SA and the data repository has no license file. `du2025.fetch()` downloads them. Without network access, the notebook falls back to the tiny synthetic case used by the tests, which has the same format and none of the benchmark's numbers.

In [1]:
import jax
import jax.numpy as jnp
import numpy as np
from pathlib import Path

from difflow_refinery.benchmarks import du2025
from difflow_refinery.benchmarks.du2025 import blocks

jax.config.update("jax_enable_x64", True)

try:
    case = du2025.find_case(1) or du2025.fetch() / "case1"
    REAL = True
except Exception as exc:                 # offline: the synthetic test case
    print("benchmark data unavailable:", exc)
    case, REAL = Path("../tests/refinery/du2025_mini/case1"), False

data = du2025.load_case(case)
model = du2025.build_model(data)
print(f"{'Case 1' if REAL else 'synthetic case'}: {model.n} variables, "
      f"{model.eq.n_rows} equations, {model.ineq.n_rows} inequalities")

Case 1: 1850 variables, 1621 equations, 81 inequalities


## The published optimum

The paper reports a profit of 34,167,968 for Case 1, and BARON and ANTIGONE agree on it. The solution itself is a GAMS `.gdx` file. `load_solution` reads it through `gams.transfer` (`pip install gamspy-base 'gamsapi[transfer]'`), or reads a JSON export of it (`du2025.export_solution_json`).

The model reproduces the published point only after four rules that the paper does not state are added. They were read from the bounds stored with the solution. One of them makes an *untraced* property that some equation references into a free variable. Those free variables can take unphysical values at the optimum, so do not read plant data from them.

In [2]:
sol = None
for name in ("solution_baron.json", "solution_baron.gdx"):
    try:
        sol = du2025.load_solution(case / name)
        break
    except (FileNotFoundError, ImportError) as exc:
        print(f"{name}: {type(exc).__name__}")

if sol is not None:
    x_pub = model.x_from_solution(sol)
    print("violations at the published point:", model.violations(x_pub))
    print(f"profit {float(model.profit(x_pub)):,.2f}  (paper: {du2025.CASE1_PUBLISHED_PROFIT:,.0f})")
    print(f"{len(model.untracked)} untraced properties are free variables, e.g.",
          model.untracked[:3])

violations at the published point: {'eq': 8.460807876531362e-08, 'ineq': 5.684341886080802e-14, 'bounds': 0.0}
profit 34,167,967.96  (paper: 34,167,968)
21 untraced properties are free variables, e.g. (('s6', 'Q0'), ('s170', 'Q37'), ('s170', 'Q51'))


## A local solve

With denominators cleared, every equation is at most bilinear. The model stores itself as sparse polynomial coefficients, so its Jacobian sparsity pattern is exact and needs no probing. `solve_local` hands the model to pounce. Pooling makes the problem nonconvex, so a local solve finds *a* local optimum. Here it starts at the published point, or at a point built from the blocks on the synthetic case.

In [3]:
x0 = x_pub if sol is not None else np.clip(np.ones(model.n), model.lower, model.upper)
xs, info = du2025.solve_local(model, x0, options=du2025.WARM_START_OPTIONS)
print(info["status_msg"], f"profit {info['profit']:,.2f}", model.violations(xs))

 INFO pounce::linesearch: theta_max raised 6.748e5 -> 6.748e7 (raise 1 of 4): 3 consecutive line searches had every trial rejected at the theta_max gate (pounce#546)


Solve_Succeeded profit 34,167,968.55 {'eq': 2.9103830456733704e-11, 'ineq': 9.044256898960157e-09, 'bounds': 1.084587086808142e-07}


## Which limits bind

The constraint multipliers give the marginal profit of each inequality, in the solver's scaled units, which are millions per unit. The largest ones show which capacities and specifications limit the plan.

In [4]:
mult = np.asarray(info["mult_g"])[model.eq.n_rows:] / du2025.model.PROFIT_SCALE
g = np.asarray(model.ineq(xs))
order = np.argsort(-np.abs(mult))[:8]
for i in order:
    print(f"{str(model.ineq.names[i]):40s} slack {-g[i]:9.2e}   marginal profit {abs(mult[i]):12,.0f}")

('25', 'ccap6', 'min')                   slack -9.04e-09   marginal profit    1,046,306
('25', 'ccap5', 'min')                   slack -8.62e-09   marginal profit      723,099
('26', 'ccap2', 'min')                   slack -7.99e-09   marginal profit      498,245
('25', 'ccap10', 'max')                  slack -5.77e-09   marginal profit      236,159
('25', 'ccap19', 'max')                  slack -4.65e-09   marginal profit      186,911
('25', 'ccap13', 'max')                  slack -2.61e-09   marginal profit      135,397
('25', 'ccap4', 'min')                   slack -2.52e-09   marginal profit      133,641
('19', 'Ublend10', 'Q14', 'min')         slack -0.00e+00   marginal profit      100,000


## Differentiating a delta-base unit

A delta-base unit shifts its yields linearly with the feed properties (Eq. 9) and renormalizes them (Eq. 10). Its Jacobian with respect to the feed properties at the base point is the published delta vector `delta / Del`, scaled by the feed. #533 compares this Jacobian with the AD Jacobian of a nonlinear unit. Here it is `Upd0` (a diesel hydrotreater), batch `m22`, with its feed at the base properties.

In [5]:
if REAL:
    u, m = "Upd0", "m22"
    ins = [s for (uu, mm, s) in data.sets["IM"] if (uu, mm) == (u, m)]
    outs = [s for (uu, mm, s) in data.sets["OM"] if (uu, mm) == (u, m)]
    qs = [q for (uu, mm, s, q) in data.sets["DBSQ"] if (uu, mm) == (u, m)]
else:
    u, m, ins, outs, qs = "Upd0", "m2", ["s2"], ["s6", "s7"], ["Q0"]
P = data.param
gam = lambda ss: jnp.array([P("gamma").get((u, m, s), 0.0) for s in ss])
dlt = lambda ss: jnp.array([[P("delta").get((u, m, s, q), 0.0) for q in qs] for s in ss])
base = jnp.array([P("B")[(u, m, q)] for q in qs])
step = jnp.array([P("Del")[(u, m, q)] for q in qs])
feeds = gam(ins)                    # feed in its base proportions, 1 t of the main feed

def yields(props):
    out, _ = blocks.delta_base(feeds, gam(ins), gam(outs), props, base, dlt(ins), dlt(outs), step)
    return out

J = jax.jacobian(yields)(base)
print("properties:", qs)
for s, row, d in zip(outs, np.asarray(J), np.asarray(dlt(outs) / step)):
    print(f"{s:6s} dF/dFQ {np.round(row, 6)}   delta/Del {np.round(d, 6)}")

properties: ['Q0', 'Q168']
s140   dF/dFQ [ 1.088e-03 -1.000e-06]   delta/Del [0.00109 0.     ]
s227   dF/dFQ [-7.62e-04 -1.10e-05]   delta/Del [-7.6e-04 -1.0e-05]
s91    dF/dFQ [-3.27e-04  5.20e-05]   delta/Del [-0.0002   0.00013]
s126   dF/dFQ [-0. -0.]   delta/Del [0. 0.]


At the base point the two columns differ only by the renormalization of Eq. 10. That term moves every outlet when the inlet coefficients move too, as they do for the hydrogen co-feed. When the published unit is replaced by a nonlinear model, `J` is the quantity to compare.

See the docs, "The Du et al. (2025) planning benchmark", in `docs/unit-operations-refinery.md`.